# JobCompass — Notebook 05: Retrieval + Reranking

## Pipeline
```
Candidate profile / Job posting
         │
         ▼
   build_query()
   ┌──────────────────────────────────────────┐
   │  Hard filters  (seniority, location)     │  ← filter context: don't affect scoring
   │  BM25 fields   (skills, title, raw_text) │  ← must/should clauses
   │  kNN vector    (description_vector)      │  ← separate ANN query
   └──────────────────────────────────────────┘
         │
   ┌─────┴──────┐
   ▼            ▼
bm25_search() semantic_search()     ← parallel, top-100 each
   │            │
   └─────┬──────┘
         ▼
   rrf_combine()                    ← reciprocal rank fusion → top-100 merged
         │
         ▼
     rerank()                       ← bge-reranker-v2-m3 cross-encoder → top-10
         │
         ▼
  format_results()                  ← match_score, matched_skills, missing_skills
```

## Why this 4-stage architecture

**BM25** catches exact skill/keyword matches that dense vectors miss  
(e.g. a candidate with 'Kubernetes' won't always be semantically close to a job  
that lists 'k8s' — BM25 handles both via tokenisation).

**Semantic search** catches conceptual matches BM25 misses  
(e.g. 'ML engineer' vs 'machine learning scientist' — different tokens, same role).

**RRF** fuses both ranked lists purely by rank position, not score magnitude.  
This is important because BM25 scores and cosine scores are on incomparable scales.  
RRF with k=60 is the de-facto standard fusion method (Cormack et al., 2009).

**Cross-encoder reranker** sees the full (query, document) pair simultaneously,  
enabling cross-attention that bi-encoders can't do. Much higher precision on top-10  
at the cost of latency — which is acceptable since we only rerank 100 candidates.

## Outputs
- `retrieval_engine.py` — importable module for use in the API layer (NB06)
- Interactive evaluation cells to verify result quality

## Prerequisites
- OpenSearch running with data from NB04
- No vLLM server needed
- Reranker model downloads ~1.1GB on first run

In [1]:
# Cell 0 — Install dependencies
import subprocess, sys

packages = [
    "opensearch-py>=2.4.0",
    "sentence-transformers>=3.0.0",
    "torch",
    "transformers>=4.51.0",
    "orjson",
    "numpy",
]
for pkg in packages:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    print(f"  {'OK' if r.returncode == 0 else 'FAIL'} {pkg}")
print("Done.")

  OK opensearch-py>=2.4.0
  OK sentence-transformers>=3.0.0
  OK torch
  OK transformers>=4.51.0
  OK orjson
  OK numpy
Done.


In [6]:
# Cell 1 — Configuration
import os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── OpenSearch ────────────────────────────────────────────────────────────────
OS_HOST           = "localhost"
OS_PORT           = 9200
CANDIDATES_ALIAS  = "candidates"
JOBS_ALIAS        = "jobs"

# ── Retrieval ─────────────────────────────────────────────────────────────────
# How many candidates BM25 and semantic search each retrieve before fusion.
# More = better recall before reranking, at the cost of reranker latency.
BM25_TOP_K        = 100
SEMANTIC_TOP_K    = 100
RRF_K             = 60      # RRF constant. 60 is the standard value from the original paper.
RERANK_TOP_N      = 10      # Final results returned after reranking.

# ── Embedding model (must match NB03) ─────────────────────────────────────────
EMBED_MODEL_NAME  = "Qwen/Qwen3-Embedding-0.6B"
EMBED_DIM         = 1024
EMBED_MAX_SEQ_LEN = 512
# Instructions used at query time (asymmetric: query != passage)
# At retrieval time the input IS a query, so we use the query instruction.
CANDIDATE_QUERY_INSTRUCTION = "Find jobs that match this candidate profile"
JOB_QUERY_INSTRUCTION       = "Find candidates that match this job description"

# ── Reranker model ────────────────────────────────────────────────────────────
# bge-reranker-v2-m3: multilingual cross-encoder, state-of-the-art on BEIR
# Alternatives:
#   bge-reranker-v2-gemma  — better quality, needs 8GB VRAM
#   cross-encoder/ms-marco-MiniLM-L-6-v2 — faster, English-only, lower quality
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

# ── Cache ─────────────────────────────────────────────────────────────────────
HF_HOME = "/mnt/d/huggingface_cache"
os.environ["HF_HOME"] = HF_HOME

# ── Output module path ────────────────────────────────────────────────────────
MODULE_OUTPUT_PATH = "../src/retrieval_engine.py"
os.makedirs("../src", exist_ok=True)

print("Config ready.")
print(f"  OpenSearch       : {OS_HOST}:{OS_PORT}")
print(f"  BM25 top-k       : {BM25_TOP_K}")
print(f"  Semantic top-k   : {SEMANTIC_TOP_K}")
print(f"  RRF k            : {RRF_K}")
print(f"  Rerank top-n     : {RERANK_TOP_N}")
print(f"  Embed model      : {EMBED_MODEL_NAME}")
print(f"  Reranker         : {RERANKER_MODEL_NAME}")

Config ready.
  OpenSearch       : localhost:9200
  BM25 top-k       : 100
  Semantic top-k   : 100
  RRF k            : 60
  Rerank top-n     : 10
  Embed model      : Qwen/Qwen3-Embedding-0.6B
  Reranker         : BAAI/bge-reranker-v2-m3


In [4]:
# Cell 2 — Connect to OpenSearch
from opensearchpy import OpenSearch, RequestsHttpConnection

client = OpenSearch(
    hosts            = [{"host": OS_HOST, "port": OS_PORT}],
    http_compress    = True,
    use_ssl          = False,
    verify_certs     = False,
    ssl_show_warn    = False,
    connection_class = RequestsHttpConnection,
    timeout          = 30,
    max_retries      = 3,
    retry_on_timeout = True,
)

info = client.info()
print(f"OpenSearch : connected ✓  (v{info['version']['number']})")

for alias in [CANDIDATES_ALIAS, JOBS_ALIAS]:
    count = client.count(index=alias)["count"]
    print(f"  '{alias}' : {count:,} documents")
    if count == 0:
        raise RuntimeError(f"Index '{alias}' is empty — run NB04 first.")

OpenSearch : connected ✓  (v2.13.0)
  'candidates' : 2,754 documents
  'jobs' : 5,000 documents


In [7]:
# Cell 3 — Load embedding model
#
# The embedding model is used here to encode the QUERY at retrieval time.
# We use a different instruction than NB03 (passage indexing) because
# Qwen3-Embedding is an asymmetric model trained with task instructions.
# At query time, we're asking "find me similar documents" — that's a query.
# At index time (NB03), we were encoding the documents themselves — those are passages.

import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
print(f"Loading {EMBED_MODEL_NAME} ...")

embed_tokenizer = AutoTokenizer.from_pretrained(
    EMBED_MODEL_NAME, cache_dir=HF_HOME
)
embed_model = AutoModel.from_pretrained(
    EMBED_MODEL_NAME,
    cache_dir   = HF_HOME,
    torch_dtype = torch.float16,
    device_map  = "auto",
).eval()


def _last_token_pool(last_hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    left_padded = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padded:
        return last_hidden[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    return last_hidden[
        torch.arange(last_hidden.shape[0], device=last_hidden.device), seq_lens
    ]


def encode_query(text: str, instruction: str) -> np.ndarray:
    """
    Encode a single query text into a unit-norm vector.
    Uses the query-side instruction (different from NB03's passage instruction).
    Returns float32 numpy array of shape (EMBED_DIM,).
    """
    formatted = f"Instruct: {instruction}\nQuery: {text.strip()}"
    inputs = embed_tokenizer(
        [formatted],
        max_length     = EMBED_MAX_SEQ_LEN,
        padding        = True,
        truncation     = True,
        return_tensors = "pt",
    ).to(embed_model.device)

    with torch.no_grad():
        out = embed_model(**inputs)

    vec = _last_token_pool(out.last_hidden_state, inputs["attention_mask"])
    vec = F.normalize(vec, p=2, dim=1)
    return vec.cpu().float().numpy()[0]


# Smoke test
test_vec = encode_query("python developer 5 years", CANDIDATE_QUERY_INSTRUCTION)
assert test_vec.shape == (EMBED_DIM,)
assert 0.99 < np.linalg.norm(test_vec) < 1.01
print(f"Embedding model loaded ✓  shape={test_vec.shape}")

Device : cuda
Loading Qwen/Qwen3-Embedding-0.6B ...
Embedding model loaded ✓  shape=(1024,)


In [8]:
# Cell 4 — Load reranker model
#
# Cross-encoder architecture: takes (query_text, document_text) as a single
# concatenated input and outputs a single relevance score.
# Unlike bi-encoders, cross-attention is performed between query and document,
# enabling much finer-grained relevance judgment.
#
# Trade-off: O(n) inference for n candidates, but n=100 is fast enough.
# On GPU: ~50ms for 100 pairs. On CPU: ~2s for 100 pairs.

from sentence_transformers import CrossEncoder

print(f"Loading {RERANKER_MODEL_NAME} ...")
print("(Downloads ~1.1GB on first run)")

reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    max_length   = 512,
    device       = device,
    cache_folder = HF_HOME,
)

# Smoke test
test_scores = reranker.predict([
    ("python developer", "We are looking for a Python engineer with Django experience"),
    ("python developer", "We need a plumber for residential repairs"),
])
assert test_scores[0] > test_scores[1], "Reranker sanity check failed"
print(f"Reranker loaded ✓")
print(f"  Relevance scores: relevant={test_scores[0]:.4f}  irrelevant={test_scores[1]:.4f}  (higher=more relevant)")

Loading BAAI/bge-reranker-v2-m3 ...
(Downloads ~1.1GB on first run)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Reranker loaded ✓
  Relevance scores: relevant=0.3832  irrelevant=0.0000  (higher=more relevant)


In [9]:
# Cell 5 — Query builders
#
# Two query builders, one per direction:
#   find_jobs_for_candidate()  — given a candidate profile, find matching jobs
#   find_candidates_for_job()  — given a job, find matching candidates
#
# Both return an (embed_text, bm25_query_dsl, filters) tuple used by the
# search functions in Cell 6.
#
# FILTER CONTEXT vs QUERY CONTEXT
# Filters go in the OpenSearch 'filter' clause, not 'must'/'should'.
# Filtered documents are excluded entirely from results but filters do NOT
# affect BM25 scores for the remaining documents.
# This is correct behaviour: a Python developer with 3 years experience
# is not MORE relevant to a senior role just because seniority filtering is off.

from typing import Optional


def build_job_search_query(
    candidate        : dict,
    filter_seniority : Optional[list[str]] = None,
    filter_location  : Optional[str]       = None,
    filter_work_type : Optional[str]       = None,
) -> tuple[str, dict, list]:
    """
    Build search components for finding jobs that match a candidate profile.

    Returns:
        embed_text   : text to encode for semantic search
        bm25_body    : OpenSearch query DSL for BM25 search
        filters      : list of OpenSearch filter clauses
    """
    # Text for semantic query vector
    embed_text = (candidate.get("embedding_text") or "").strip()
    if not embed_text:
        parts = []
        if candidate.get("current_title"): parts.append(candidate["current_title"])
        flat = candidate.get("skills_flat") or []
        if flat: parts.append("Skills: " + ", ".join(flat[:20]))
        embed_text = "\n".join(parts)

    # BM25 query: skills are most important, then title, then full text
    skills_flat = candidate.get("skills_flat") or []
    title       = candidate.get("current_title") or ""
    summary     = candidate.get("summary") or ""

    should_clauses = []

    # Skills match — highest weight
    if skills_flat:
        should_clauses.append({
            "match": {
                "required_skills": {
                    "query"   : " ".join(skills_flat[:30]),
                    "boost"   : 3.0,
                    "operator": "or",
                }
            }
        })
        should_clauses.append({
            "match": {
                "skills_flat": {
                    "query" : " ".join(skills_flat[:30]),
                    "boost" : 2.5,
                }
            }
        })

    # Title match
    if title:
        should_clauses.append({
            "match": {
                "normalized_title": {
                    "query": title,
                    "boost": 2.0,
                }
            }
        })

    # Summary / experience context match
    if summary:
        should_clauses.append({
            "match": {
                "description": {
                    "query": summary[:300],
                    "boost": 1.0,
                }
            }
        })

    bm25_body = {
        "query": {
            "bool": {
                "should"              : should_clauses,
                "minimum_should_match": 1,
            }
        }
    }

    # Hard filters — applied in filter context (no score impact)
    filters = []
    if filter_seniority:
        filters.append({"terms": {"seniority": filter_seniority}})
    if filter_location:
        filters.append({
            "bool": {"should": [
                {"term": {"location.city"   : filter_location}},
                {"term": {"location.country": filter_location}},
            ]}
        })
    if filter_work_type:
        filters.append({"term": {"work_type": filter_work_type}})

    # Only show active jobs
    filters.append({"term": {"is_active": True}})

    return embed_text, bm25_body, filters


def build_candidate_search_query(
    job              : dict,
    filter_seniority : Optional[list[str]] = None,
    filter_location  : Optional[str]       = None,
) -> tuple[str, dict, list]:
    """
    Build search components for finding candidates that match a job posting.
    Symmetric to build_job_search_query.
    """
    embed_text = (job.get("embedding_text") or "").strip()
    if not embed_text:
        parts = []
        if job.get("normalized_title"): parts.append(job["normalized_title"])
        skills = job.get("required_skills") or job.get("skills_flat") or []
        if skills: parts.append("Skills: " + ", ".join(skills[:20]))
        embed_text = "\n".join(parts)

    required_skills = job.get("required_skills") or job.get("skills_flat") or []
    title           = job.get("normalized_title") or job.get("title") or ""
    summary         = job.get("responsibilities_summary") or ""

    should_clauses = []

    if required_skills:
        should_clauses.append({
            "match": {
                "skills_flat": {
                    "query"   : " ".join(required_skills[:30]),
                    "boost"   : 3.0,
                    "operator": "or",
                }
            }
        })
        should_clauses.append({
            "match": {
                "skills.technical": {
                    "query": " ".join(required_skills[:30]),
                    "boost": 2.5,
                }
            }
        })

    if title:
        should_clauses.append({
            "match": {
                "current_title": {
                    "query": title,
                    "boost": 2.0,
                }
            }
        })

    if summary:
        should_clauses.append({
            "match": {
                "raw_text": {
                    "query": summary[:300],
                    "boost": 1.0,
                }
            }
        })

    bm25_body = {
        "query": {
            "bool": {
                "should"              : should_clauses,
                "minimum_should_match": 1,
            }
        }
    }

    filters = []
    if filter_seniority:
        filters.append({"terms": {"seniority": filter_seniority}})
    if filter_location:
        filters.append({
            "bool": {"should": [
                {"term": {"location.city"   : filter_location}},
                {"term": {"location.country": filter_location}},
            ]}
        })

    return embed_text, bm25_body, filters


print("Query builders defined.")

Query builders defined.


In [10]:
# Cell 6 — BM25 search + semantic search

def bm25_search(
    index_alias : str,
    bm25_body   : dict,
    filters     : list,
    top_k       : int = BM25_TOP_K,
    source_fields: list[str] | None = None,
) -> list[dict]:
    """
    Execute BM25 lexical search with optional hard filters.
    Returns list of {id, score, source} dicts.
    """
    query = bm25_body["query"]

    # Wrap in bool with filter context if filters exist
    if filters:
        query = {
            "bool": {
                "must"  : [query],
                "filter": filters,
            }
        }

    body = {
        "query"  : query,
        "size"   : top_k,
        "_source": source_fields if source_fields else True,
    }

    resp = client.search(index=index_alias, body=body)
    return [
        {"id": h["_id"], "score": h["_score"], "source": h["_source"]}
        for h in resp["hits"]["hits"]
    ]


def semantic_search(
    index_alias : str,
    embed_text  : str,
    instruction : str,
    filters     : list,
    top_k       : int = SEMANTIC_TOP_K,
    source_fields: list[str] | None = None,
) -> list[dict]:
    """
    Execute approximate nearest-neighbour search using the Lucene HNSW index.
    Returns list of {id, score, source} dicts.

    kNN and filter context:
    OpenSearch 2.9+ supports pre-filtering for kNN queries. The filter is applied
    before ANN traversal, which gives exact filtered results.
    Pre-filtering (efficient_filter) is the default in OS 2.9+.
    """
    query_vector = encode_query(embed_text, instruction)

    knn_clause = {
        "description_vector": {
            "vector": query_vector.tolist(),
            "k"     : top_k,
        }
    }

    # Add pre-filters to kNN query if filters exist
    if filters:
        knn_clause["description_vector"]["filter"] = {
            "bool": {"filter": filters}
        }

    body = {
        "size" : top_k,
        "query": {"knn": knn_clause},
        "_source": source_fields if source_fields else True,
    }

    resp = client.search(index=index_alias, body=body)
    return [
        {"id": h["_id"], "score": h["_score"], "source": h["_source"]}
        for h in resp["hits"]["hits"]
    ]


print("Search functions defined.")

Search functions defined.


In [ ]:
# Cell 7 — Reciprocal Rank Fusion
#
# RRF formula (Cormack et al., 2009):
#   RRF_score(d) = Σ_r  1 / (k + rank_r(d))
#
# where:
#   d       = document
#   r       = retrieval system (BM25, semantic)
#   rank_r  = rank position of d in system r (1-indexed, lower = better)
#   k       = smoothing constant (60 is universally recommended)
#
# Why RRF instead of score normalisation:
#   BM25 scores are unbounded and depend on corpus statistics.
#   Cosine similarity scores are in [0,1] but their distribution varies.
#   Normalising and combining incomparable scales is fragile.
#   RRF only uses rank order, which is stable across both systems.
#
# A document in position 1 in both lists gets:
#   1/(60+1) + 1/(60+1) ≈ 0.0328 — highest possible score
# A document in position 100 in both lists gets:
#   1/(60+100) + 1/(60+100) ≈ 0.0125 — lowest score

def rrf_combine(
    bm25_results    : list[dict],
    semantic_results: list[dict],
    k               : int = RRF_K,
    top_n           : int = 100,
) -> list[dict]:
    """
    Fuse BM25 and semantic ranked lists using Reciprocal Rank Fusion.

    Args:
        bm25_results     : list of {id, score, source} from bm25_search()
        semantic_results : list of {id, score, source} from semantic_search()
        k                : RRF constant (default 60)
        top_n            : number of results to return after fusion

    Returns:
        list of {id, rrf_score, bm25_rank, semantic_rank, source}
        sorted by rrf_score descending
    """
    scores   : dict[str, float]  = {}
    bm25_rank: dict[str, int]    = {}
    sem_rank : dict[str, int]    = {}
    sources  : dict[str, dict]   = {}

    # Assign rank-based scores from BM25 list
    for rank, hit in enumerate(bm25_results, start=1):
        doc_id = hit["id"]
        scores[doc_id]    = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
        bm25_rank[doc_id] = rank
        sources[doc_id]   = hit["source"]

    # Assign rank-based scores from semantic list
    for rank, hit in enumerate(semantic_results, start=1):
        doc_id = hit["id"]
        scores[doc_id]   = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
        sem_rank[doc_id] = rank
        if doc_id not in sources:
            sources[doc_id] = hit["source"]

    # Sort by fused score descending, take top_n
    ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_n]

    return [
        {
            "id"            : doc_id,
            "rrf_score"     : round(score, 6),
            "bm25_rank"     : bm25_rank.get(doc_id),    # None if not in BM25 results
            "semantic_rank" : sem_rank.get(doc_id),      # None if not in semantic results
            "source"        : sources[doc_id],
        }
        for doc_id, score in ranked
    ]

print("rrf_combine() defined.")

rrf_combine() defined.


In [13]:
# Cell 8 — Cross-encoder reranking

def build_rerank_text_candidate(source: dict) -> str:
    """
    Build a compact text representation of a candidate for the reranker.
    The reranker sees (query_text, this_text) as a pair.
    Keep it under 256 tokens — the reranker has a 512 token limit total.
    """
    parts = []
    if source.get("current_title"):      parts.append(f"Title: {source['current_title']}")
    if source.get("seniority"):          parts.append(f"Seniority: {source['seniority']}")
    skills = source.get("skills_flat") or []
    if skills:                           parts.append(f"Skills: {', '.join(skills[:20])}")
    if source.get("summary"):            parts.append(source["summary"][:200])
    exp = source.get("work_experience") or []
    for e in exp[:2]:
        parts.append(f"{e.get('title','')} at {e.get('company','')} — {e.get('description','')[:100]}")
    return "\n".join(parts)[:800]


def build_rerank_text_job(source: dict) -> str:
    """
    Build a compact text representation of a job for the reranker.
    """
    parts = []
    if source.get("normalized_title"):   parts.append(f"Title: {source['normalized_title']}")
    if source.get("seniority"):          parts.append(f"Seniority: {source['seniority']}")
    skills = source.get("required_skills") or source.get("skills_flat") or []
    if skills:                           parts.append(f"Required skills: {', '.join(skills[:20])}")
    if source.get("responsibilities_summary"):
        parts.append(source["responsibilities_summary"][:300])
    elif source.get("description"):
        parts.append(source["description"][:300])
    return "\n".join(parts)[:800]


def rerank(
    query_text  : str,
    candidates  : list[dict],
    text_builder,
    top_n       : int = RERANK_TOP_N,
) -> list[dict]:
    """
    Rerank a list of RRF-fused candidates using the cross-encoder.

    Args:
        query_text   : the text representing what we're searching for
        candidates   : list of dicts from rrf_combine()
        text_builder : function that converts a source dict to reranker input text
        top_n        : number of final results to return

    Returns:
        list of dicts with added 'rerank_score' field, sorted by rerank_score desc
    """
    if not candidates:
        return []

    # Build (query, document) pairs for the cross-encoder
    pairs = [
        (query_text, text_builder(c["source"]))
        for c in candidates
    ]

    # Score all pairs in one batch
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

    # Attach scores and sort
    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    return sorted(candidates, key=lambda x: -x["rerank_score"])[:top_n]


print("Reranking functions defined.")

Reranking functions defined.


In [14]:
# Cell 9 — Result formatting
#
# Computes matched_skills and missing_skills by comparing the query's
# skills_flat against the result's skills_flat (or required_skills for jobs).
# Both lists are lowercased for comparison to handle capitalisation differences.

def compute_skill_overlap(
    query_skills  : list[str],
    result_skills : list[str],
) -> tuple[list[str], list[str]]:
    """
    Returns (matched_skills, missing_skills) relative to query_skills.
    matched = skills the result has that the query also has
    missing = skills the query has that the result is missing
    """
    q_lower = {s.lower() for s in query_skills}
    r_lower = {s.lower() for s in result_skills}
    matched = [s for s in query_skills  if s.lower() in r_lower]
    missing = [s for s in query_skills  if s.lower() not in r_lower]
    return matched, missing


def format_job_results(
    reranked_hits : list[dict],
    candidate     : dict,
) -> list[dict]:
    """
    Format final job results for a candidate.
    Computes per-result skill overlap and a normalised match_score.
    """
    cand_skills    = candidate.get("skills_flat") or []
    max_rr_score   = reranked_hits[0]["rerank_score"] if reranked_hits else 1.0

    results = []
    for rank, hit in enumerate(reranked_hits, start=1):
        src            = hit["source"]
        job_skills     = src.get("required_skills") or src.get("skills_flat") or []
        matched, missing = compute_skill_overlap(cand_skills, job_skills)

        # Normalised match score: rerank_score / max_rerank_score → [0,1]
        match_score = round(
            hit["rerank_score"] / max(abs(max_rr_score), 1e-6), 4
        ) if max_rr_score != 0 else 0.0

        results.append({
            "rank"            : rank,
            "job_id"          : hit["id"],
            "title"           : src.get("normalized_title") or src.get("title"),
            "company"         : src.get("company"),
            "location"        : src.get("location", {}),
            "seniority"       : src.get("seniority"),
            "work_type"       : src.get("work_type"),
            "salary"          : src.get("salary", {}),
            "domain_tags"     : src.get("domain_tags", []),
            "match_score"     : match_score,
            "rerank_score"    : round(hit["rerank_score"], 4),
            "rrf_score"       : hit["rrf_score"],
            "bm25_rank"       : hit["bm25_rank"],
            "semantic_rank"   : hit["semantic_rank"],
            "matched_skills"  : matched[:10],
            "missing_skills"  : missing[:10],
            "responsibilities_summary": src.get("responsibilities_summary"),
        })
    return results


def format_candidate_results(
    reranked_hits : list[dict],
    job           : dict,
) -> list[dict]:
    """
    Format final candidate results for a job.
    """
    job_skills   = job.get("required_skills") or job.get("skills_flat") or []
    max_rr_score = reranked_hits[0]["rerank_score"] if reranked_hits else 1.0

    results = []
    for rank, hit in enumerate(reranked_hits, start=1):
        src            = hit["source"]
        cand_skills    = src.get("skills_flat") or []
        matched, missing = compute_skill_overlap(job_skills, cand_skills)

        match_score = round(
            hit["rerank_score"] / max(abs(max_rr_score), 1e-6), 4
        ) if max_rr_score != 0 else 0.0

        results.append({
            "rank"            : rank,
            "candidate_id"    : hit["id"],
            "name"            : src.get("name"),
            "current_title"   : src.get("current_title"),
            "seniority"       : src.get("seniority"),
            "location"        : src.get("location", {}),
            "total_years_exp" : src.get("total_years_experience"),
            "match_score"     : match_score,
            "rerank_score"    : round(hit["rerank_score"], 4),
            "rrf_score"       : hit["rrf_score"],
            "bm25_rank"       : hit["bm25_rank"],
            "semantic_rank"   : hit["semantic_rank"],
            "matched_skills"  : matched[:10],
            "missing_skills"  : missing[:10],
            "profile_quality" : src.get("profile_quality_score"),
        })
    return results


print("Result formatting functions defined.")

Result formatting functions defined.


In [15]:
# Cell 10 — Full pipeline functions
#
# Two high-level functions that run the complete retrieval pipeline:
#   find_jobs_for_candidate(candidate_dict, **filters) → top-10 jobs
#   find_candidates_for_job(job_dict, **filters)       → top-10 candidates

import time


def find_jobs_for_candidate(
    candidate        : dict,
    filter_seniority : Optional[list[str]] = None,
    filter_location  : Optional[str]       = None,
    filter_work_type : Optional[str]       = None,
    bm25_top_k       : int                 = BM25_TOP_K,
    semantic_top_k   : int                 = SEMANTIC_TOP_K,
    rerank_top_n     : int                 = RERANK_TOP_N,
    verbose          : bool                = False,
) -> list[dict]:
    """
    Full retrieval pipeline: given a candidate profile, return top matching jobs.

    Pipeline: build_query → bm25_search + semantic_search → rrf_combine → rerank → format
    """
    t0 = time.time()

    embed_text, bm25_body, filters = build_job_search_query(
        candidate, filter_seniority, filter_location, filter_work_type
    )

    # Retrieve from both systems
    bm25_hits     = bm25_search(JOBS_ALIAS, bm25_body, filters, top_k=bm25_top_k)
    semantic_hits = semantic_search(
        JOBS_ALIAS, embed_text, CANDIDATE_QUERY_INSTRUCTION, filters, top_k=semantic_top_k
    )

    if verbose:
        print(f"  BM25: {len(bm25_hits)} hits | Semantic: {len(semantic_hits)} hits")

    # Fuse
    fused = rrf_combine(bm25_hits, semantic_hits)

    if verbose:
        print(f"  After RRF: {len(fused)} unique candidates")

    # Rerank
    query_text   = embed_text
    reranked     = rerank(query_text, fused, build_rerank_text_job, top_n=rerank_top_n)

    # Format
    results = format_job_results(reranked, candidate)

    if verbose:
        elapsed = time.time() - t0
        print(f"  Total latency: {elapsed*1000:.0f}ms")

    return results


def find_candidates_for_job(
    job              : dict,
    filter_seniority : Optional[list[str]] = None,
    filter_location  : Optional[str]       = None,
    bm25_top_k       : int                 = BM25_TOP_K,
    semantic_top_k   : int                 = SEMANTIC_TOP_K,
    rerank_top_n     : int                 = RERANK_TOP_N,
    verbose          : bool                = False,
) -> list[dict]:
    """
    Full retrieval pipeline: given a job posting, return top matching candidates.
    """
    t0 = time.time()

    embed_text, bm25_body, filters = build_candidate_search_query(
        job, filter_seniority, filter_location
    )

    bm25_hits     = bm25_search(CANDIDATES_ALIAS, bm25_body, filters, top_k=bm25_top_k)
    semantic_hits = semantic_search(
        CANDIDATES_ALIAS, embed_text, JOB_QUERY_INSTRUCTION, filters, top_k=semantic_top_k
    )

    if verbose:
        print(f"  BM25: {len(bm25_hits)} hits | Semantic: {len(semantic_hits)} hits")

    fused    = rrf_combine(bm25_hits, semantic_hits)
    reranked = rerank(embed_text, fused, build_rerank_text_candidate, top_n=rerank_top_n)
    results  = format_candidate_results(reranked, job)

    if verbose:
        elapsed = time.time() - t0
        print(f"  Total latency: {elapsed*1000:.0f}ms")

    return results


print("Pipeline functions defined.")

Pipeline functions defined.


In [16]:
# Cell 11 — Evaluation: jobs for a candidate
import orjson, random

# Load a sample candidate from the embedded file
candidates_path = "../data/processed/candidates_embedded.jsonl"
sample_candidates = []
with open(candidates_path, "rb") as f:
    for line in f:
        if line.strip():
            r = orjson.loads(line)
            # Pick candidates with reasonable quality
            if r.get("profile_quality_score", 0) >= 40 and r.get("skills_flat"):
                sample_candidates.append(r)
        if len(sample_candidates) >= 50:
            break

if not sample_candidates:
    raise RuntimeError("No quality candidates found — check candidates_embedded.jsonl")

test_candidate = random.choice(sample_candidates)

print(f"Test candidate:")
print(f"  ID       : {test_candidate['candidate_id']}")
print(f"  Name     : {test_candidate.get('name')}")
print(f"  Title    : {test_candidate.get('current_title')}")
print(f"  Seniority: {test_candidate.get('seniority')}")
print(f"  Skills   : {test_candidate.get('skills_flat', [])[:8]}")
print(f"  Quality  : {test_candidate.get('profile_quality_score')}")
print()

job_results = find_jobs_for_candidate(test_candidate, verbose=True)

print(f"\nTop {len(job_results)} matching jobs:")
print(f"{'Rank':>4}  {'Title':35s}  {'Company':25s}  {'Match':>6}  {'Rerank':>7}  BM25  Sem  {'Matched skills'}")
print("-" * 130)
for r in job_results:
    print(
        f"  {r['rank']:>2}  "
        f"{str(r['title'] or '')[:34]:35s}  "
        f"{str(r['company'] or '')[:24]:25s}  "
        f"{r['match_score']:>6.3f}  "
        f"{r['rerank_score']:>7.3f}  "
        f"{str(r['bm25_rank'] or '-'):>4}  "
        f"{str(r['semantic_rank'] or '-'):>3}  "
        f"{r['matched_skills'][:4]}"
    )

Test candidate:
  ID       : 0017
  Name     : Robert Smith
  Title    : Senior General Accountant
  Seniority: senior
  Skills   : ['Payroll', 'Accounts Receivable', 'Human Resources', 'Office Management', 'Office Applications', 'Financial Reporting', 'Quickbooks', 'MAS 90']
  Quality  : 72

  BM25: 100 hits | Semantic: 100 hits
  After RRF: 100 unique candidates
  Total latency: 15428ms

Top 10 matching jobs:
Rank  Title                                Company                     Match   Rerank  BM25  Sem  Matched skills
----------------------------------------------------------------------------------------------------------------------------------
   1  Financial Accountant                 Huntsman                    1.000    0.657     1   12  ['Financial Reporting']
   2  Financial Accountant                 Merck                       1.000    0.657     2   13  ['Financial Reporting']
   3  Financial Accountant                 Campbell Soup               1.000    0.657     3   14 

In [17]:
# Cell 12 — Evaluation: candidates for a job
import orjson, random

jobs_path = "../data/processed/jobs_embedded.jsonl"
sample_jobs = []
with open(jobs_path, "rb") as f:
    for line in f:
        if line.strip():
            r = orjson.loads(line)
            if r.get("skills_flat") and r.get("description"):
                sample_jobs.append(r)
        if len(sample_jobs) >= 50:
            break

test_job = random.choice(sample_jobs)

print(f"Test job:")
print(f"  ID               : {test_job['job_id']}")
print(f"  Title            : {test_job.get('normalized_title')}")
print(f"  Company          : {test_job.get('company')}")
print(f"  Seniority        : {test_job.get('seniority')}")
print(f"  Required skills  : {test_job.get('required_skills', [])[:8]}")
print()

cand_results = find_candidates_for_job(test_job, verbose=True)

print(f"\nTop {len(cand_results)} matching candidates:")
print(f"{'Rank':>4}  {'Title':35s}  {'Seniority':10s}  {'Match':>6}  {'Rerank':>7}  BM25  Sem  {'Matched skills'}")
print("-" * 130)
for r in cand_results:
    print(
        f"  {r['rank']:>2}  "
        f"{str(r['current_title'] or '')[:34]:35s}  "
        f"{str(r['seniority'] or '')[:9]:10s}  "
        f"{r['match_score']:>6.3f}  "
        f"{r['rerank_score']:>7.3f}  "
        f"{str(r['bm25_rank'] or '-'):>4}  "
        f"{str(r['semantic_rank'] or '-'):>3}  "
        f"{r['matched_skills'][:4]}"
    )

Test job:
  ID               : job_0038
  Title            : UX/UI Designer
  Company          : Leidos Holdings
  Seniority        : mid
  Required skills  : ['UI design principles and best practices', 'Graphic design tools (e.g., Adobe Photoshop, Illustrator', 'Typography and color theory', 'Visual design and layout', 'Responsive design']

  BM25: 100 hits | Semantic: 100 hits
  Total latency: 4511ms

Top 10 matching candidates:
Rank  Title                                Seniority    Match   Rerank  BM25  Sem  Matched skills
----------------------------------------------------------------------------------------------------------------------------------
   1                                       fresher      1.000    0.773     -   57  []
   2                                       mid          0.797    0.616     -   34  []
   3                                       lead         0.671    0.519     -    6  []
   4                                       lead         0.671    0.519     -  

In [18]:
# Cell 13 — Ablation: BM25 only vs Semantic only vs Hybrid
# Shows the value of each retrieval stage by comparing top-5 results.

print(f"Ablation on candidate: {test_candidate.get('current_title')} ({test_candidate['candidate_id']})")
print(f"Skills: {test_candidate.get('skills_flat', [])[:6]}")
print()

embed_text, bm25_body, filters = build_job_search_query(test_candidate)

bm25_hits     = bm25_search(JOBS_ALIAS, bm25_body, filters, top_k=10)
semantic_hits = semantic_search(
    JOBS_ALIAS, embed_text, CANDIDATE_QUERY_INSTRUCTION, filters, top_k=10
)
fused         = rrf_combine(bm25_hits, semantic_hits, top_n=10)

def show_list(label, hits, title_key):
    print(f"  {label}:")
    for i, h in enumerate(hits[:5], 1):
        src   = h["source"]
        title = src.get(title_key) or src.get("title") or src.get("normalized_title") or "?"
        score = h.get("rrf_score") or h.get("score", 0)
        print(f"    {i}. [{score:.4f}] {str(title)[:50]}")
    print()

show_list("BM25 only (top-5)",     bm25_hits,     "normalized_title")
show_list("Semantic only (top-5)", semantic_hits, "normalized_title")
show_list("RRF fused (top-5)",     fused,         "normalized_title")

# Show overlap — documents that appeared in BOTH lists
bm25_ids = {h["id"] for h in bm25_hits}
sem_ids  = {h["id"] for h in semantic_hits}
overlap  = bm25_ids & sem_ids
print(f"  Overlap between BM25 and semantic top-{len(bm25_hits)}: {len(overlap)} documents")
print(f"  (High overlap = BM25 and semantic agree; low overlap = they find different relevant docs)")

Ablation on candidate: Senior General Accountant (0017)
Skills: ['Payroll', 'Accounts Receivable', 'Human Resources', 'Office Management', 'Office Applications', 'Financial Reporting']

  BM25 only (top-5):
    1. [267.1997] Financial Accountant
    2. [267.1997] Financial Accountant
    3. [267.1997] Financial Accountant
    4. [267.1997] Financial Accountant
    5. [267.1997] Financial Accountant

  Semantic only (top-5):
    1. [0.8086] Financial Controller
    2. [0.8086] Financial Controller
    3. [0.8086] Financial Controller
    4. [0.8086] Financial Controller
    5. [0.8086] Financial Controller

  RRF fused (top-5):
    1. [0.0164] Financial Accountant
    2. [0.0164] Financial Controller
    3. [0.0161] Financial Accountant
    4. [0.0161] Financial Controller
    5. [0.0159] Financial Accountant

  Overlap between BM25 and semantic top-10: 0 documents
  (High overlap = BM25 and semantic agree; low overlap = they find different relevant docs)


In [19]:
# Cell 14 — Export retrieval engine as importable Python module
# This module is imported by NB06 (the API layer) without re-running this notebook.

module_code = '''
"""
JobCompass Retrieval Engine
Generated by NB05. Import this in the API layer.

Usage:
    from retrieval_engine import RetrievalEngine
    engine = RetrievalEngine()
    jobs   = engine.find_jobs_for_candidate(candidate_dict)
    cands  = engine.find_candidates_for_job(job_dict)
"""
import os
import time
import numpy as np
import torch
import torch.nn.functional as F
from typing import Optional
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import CrossEncoder
from opensearchpy import OpenSearch, RequestsHttpConnection


class RetrievalEngine:
    """
    Stateful retrieval engine. Loads models once at construction.
    Thread-safe for read-only retrieval (all models are in eval mode).
    """

    def __init__(
        self,
        os_host          : str = "localhost",
        os_port          : int = 9200,
        candidates_alias : str = "candidates",
        jobs_alias       : str = "jobs",
        embed_model_name : str = "Qwen/Qwen3-Embedding-0.6B",
        reranker_name    : str = "BAAI/bge-reranker-v2-m3",
        hf_home          : str = "D:/huggingface_cache",
        bm25_top_k       : int = 100,
        semantic_top_k   : int = 100,
        rrf_k            : int = 60,
        rerank_top_n     : int = 10,
        embed_dim        : int = 2048,
        max_seq_len      : int = 512,
    ):
        os.environ["HF_HOME"] = hf_home
        self.candidates_alias = candidates_alias
        self.jobs_alias       = jobs_alias
        self.bm25_top_k       = bm25_top_k
        self.semantic_top_k   = semantic_top_k
        self.rrf_k            = rrf_k
        self.rerank_top_n     = rerank_top_n
        self.embed_dim        = embed_dim
        self.device           = "cuda" if torch.cuda.is_available() else "cpu"

        self.client = OpenSearch(
            hosts            = [{"host": os_host, "port": os_port}],
            http_compress    = True,
            use_ssl          = False,
            verify_certs     = False,
            ssl_show_warn    = False,
            connection_class = RequestsHttpConnection,
            timeout          = 30,
            max_retries      = 3,
            retry_on_timeout = True,
        )

        self.embed_tokenizer = AutoTokenizer.from_pretrained(
            embed_model_name, cache_dir=hf_home
        )
        self.embed_model = AutoModel.from_pretrained(
            embed_model_name,
            cache_dir   = hf_home,
            torch_dtype = torch.float16,
            device_map  = "auto",
        ).eval()
        self.max_seq_len = max_seq_len

        self.reranker = CrossEncoder(
            reranker_name,
            max_length   = 512,
            device       = self.device,
            cache_folder = hf_home,
        )

    # ── Encoding ──────────────────────────────────────────────────────────────

    def _last_token_pool(self, last_hidden, attention_mask):
        if (attention_mask[:, -1].sum() == attention_mask.shape[0]):
            return last_hidden[:, -1]
        seq_lens = attention_mask.sum(dim=1) - 1
        return last_hidden[
            torch.arange(last_hidden.shape[0], device=last_hidden.device), seq_lens
        ]

    def encode_query(self, text: str, instruction: str) -> np.ndarray:
        formatted = f"Instruct: {instruction}\\nQuery: {text.strip()}"
        inputs = self.embed_tokenizer(
            [formatted], max_length=self.max_seq_len,
            padding=True, truncation=True, return_tensors="pt",
        ).to(self.embed_model.device)
        with torch.no_grad():
            out = self.embed_model(**inputs)
        vec = self._last_token_pool(out.last_hidden_state, inputs["attention_mask"])
        vec = F.normalize(vec, p=2, dim=1)
        return vec.cpu().float().numpy()[0]

    # ── Search ────────────────────────────────────────────────────────────────

    def _bm25_search(self, alias, body, filters, top_k):
        query = body["query"]
        if filters:
            query = {"bool": {"must": [query], "filter": filters}}
        resp = self.client.search(index=alias, body={"query": query, "size": top_k})
        return [{"id": h["_id"], "score": h["_score"], "source": h["_source"]}
                for h in resp["hits"]["hits"]]

    def _semantic_search(self, alias, embed_text, instruction, filters, top_k):
        vec = self.encode_query(embed_text, instruction)
        knn = {"description_vector": {"vector": vec.tolist(), "k": top_k}}
        if filters:
            knn["description_vector"]["filter"] = {"bool": {"filter": filters}}
        resp = self.client.search(index=alias, body={"size": top_k, "query": {"knn": knn}})
        return [{"id": h["_id"], "score": h["_score"], "source": h["_source"]}
                for h in resp["hits"]["hits"]]

    def _rrf(self, bm25, semantic):
        scores, bm25_r, sem_r, sources = {}, {}, {}, {}
        for rank, hit in enumerate(bm25, 1):
            d = hit["id"]
            scores[d] = scores.get(d, 0) + 1/(self.rrf_k + rank)
            bm25_r[d] = rank; sources[d] = hit["source"]
        for rank, hit in enumerate(semantic, 1):
            d = hit["id"]
            scores[d] = scores.get(d, 0) + 1/(self.rrf_k + rank)
            sem_r[d]  = rank
            if d not in sources: sources[d] = hit["source"]
        ranked = sorted(scores.items(), key=lambda x: -x[1])[:100]
        return [{"id": d, "rrf_score": round(s, 6), "bm25_rank": bm25_r.get(d),
                 "semantic_rank": sem_r.get(d), "source": sources[d]}
                for d, s in ranked]

    def _rerank(self, query_text, candidates, text_builder):
        if not candidates: return []
        pairs  = [(query_text, text_builder(c["source"])) for c in candidates]
        scores = self.reranker.predict(pairs, batch_size=32, show_progress_bar=False)
        for c, s in zip(candidates, scores): c["rerank_score"] = float(s)
        return sorted(candidates, key=lambda x: -x["rerank_score"])[:self.rerank_top_n]

    # ── Public API ────────────────────────────────────────────────────────────

    def find_jobs_for_candidate(
        self, candidate: dict,
        filter_seniority: Optional[list] = None,
        filter_location: Optional[str]   = None,
        filter_work_type: Optional[str]  = None,
    ) -> list[dict]:
        from retrieval_engine_helpers import (
            build_job_search_query, build_rerank_text_job, format_job_results
        )
        embed_text, bm25_body, filters = build_job_search_query(
            candidate, filter_seniority, filter_location, filter_work_type
        )
        filters.append({"term": {"is_active": True}})
        bm25     = self._bm25_search(self.jobs_alias, bm25_body, filters, self.bm25_top_k)
        semantic = self._semantic_search(
            self.jobs_alias, embed_text,
            "Find jobs that match this candidate profile",
            filters, self.semantic_top_k
        )
        fused    = self._rrf(bm25, semantic)
        reranked = self._rerank(embed_text, fused, build_rerank_text_job)
        return format_job_results(reranked, candidate)

    def find_candidates_for_job(
        self, job: dict,
        filter_seniority: Optional[list] = None,
        filter_location: Optional[str]   = None,
    ) -> list[dict]:
        from retrieval_engine_helpers import (
            build_candidate_search_query, build_rerank_text_candidate, format_candidate_results
        )
        embed_text, bm25_body, filters = build_candidate_search_query(
            job, filter_seniority, filter_location
        )
        bm25     = self._bm25_search(self.candidates_alias, bm25_body, filters, self.bm25_top_k)
        semantic = self._semantic_search(
            self.candidates_alias, embed_text,
            "Find candidates that match this job description",
            filters, self.semantic_top_k
        )
        fused    = self._rrf(bm25, semantic)
        reranked = self._rerank(embed_text, fused, build_rerank_text_candidate)
        return format_candidate_results(reranked, job)
'''

with open(MODULE_OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(module_code.strip())

print(f"Retrieval engine module written to: {MODULE_OUTPUT_PATH}")
print()
print("NB05 complete.")
print()
print("Next: Notebook 06 — New User Onboarding (real-time async worker)")
print("  Candidate portal: upload_resume() → run_ingestion_pipeline() → quality_gate()")
print("                    → cold_start_fallback() → index_and_notify()")
print("  Recruiter portal: submit_job_listing() → extract_job_fields() → embed_and_index()")
print("                    → match_existing_candidates() → notify_recruiter()")

Retrieval engine module written to: ../src/retrieval_engine.py

NB05 complete.

Next: Notebook 06 — New User Onboarding (real-time async worker)
  Candidate portal: upload_resume() → run_ingestion_pipeline() → quality_gate()
                    → cold_start_fallback() → index_and_notify()
  Recruiter portal: submit_job_listing() → extract_job_fields() → embed_and_index()
                    → match_existing_candidates() → notify_recruiter()
